# 1. Global mCH

Part of the **[Fig. 3 chapter](fig3.md)** — see it for the panel-by-panel map, run order, and data inputs. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)).


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
# All absolute paths below resolve from these two roots; the working directory
# is the original analysis/ folder, and repro_guard prevents any existing file
# from being overwritten when you re-run this notebook. See the book's
# 'Reproduction guide' chapter for details.
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.integration.seurat_class import SeuratIntegration

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'


In [ ]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [ ]:
indir = '/data/ENTEx/'
outdir = f'{ENTEX_ROOT}/'


In [ ]:
tmp = anndata.read_h5ad(f'{outdir}clustering/merged/L1pre/5kCG_lsi.h5ad')
tmp


In [ ]:
adata = anndata.read_h5ad(f'{outdir}clustering/merged/L1/5kCG100k3C_embed.h5ad')
adata


In [ ]:
lam = pd.read_csv(f'{indir}merged/ALL_lambda.csv.gz', header=0, index_col=0)
lam

In [ ]:
selcol = ['Tissue', 'Sample', 'Donor', 
          'Cis/Trans', 'CisLongContact', 'FinalmCReads', 
          'mCHmC', 'mCHCov', 'mCHFrac', 
          'mCGmC', 'mCGCov', 'mCGFrac', 
          'mCCCmC', 'mCCCCov', 'mCCCFrac']
meta = pd.concat([tmp.obs.loc[adata.obs.index, selcol], lam.loc[adata.obs.index]], axis=1)
meta['L1'] = adata.obs['L1'].copy() 
meta

In [ ]:
meta.groupby('Tissue')['cov'].mean().sort_values()

In [ ]:
leg = np.sort(meta['Tissue'].unique())
leg

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(10, 10), dpi=300)
for xx,ax in zip(leg, axes.flatten()):
    tmp = meta.loc[(meta['Tissue']==xx)]

    # selc = (tmp['cov']>100)
    # ax.scatter(tmp.loc[selc, 'ratio'], tmp.loc[selc, 'mCHFrac'], c='C0', s=0.5, edgecolor='none')
    # ax.scatter(tmp.loc[~selc, 'ratio'], tmp.loc[~selc, 'mCHFrac'], c='C1', s=0.5, edgecolor='none')
    cov = np.log10(tmp['cov']+1)
    # cov[cov>np.percentile(cov, 99)] = np.percentile(cov, 99)
    # cov[cov<np.percentile(cov, 1)] = np.percentile(cov, 1)
    
    ax.scatter(tmp['ratio'], tmp['mCHFrac'], c=cov, cmap='bwr', s=0.5, 
               vmin=0, vmax=4, edgecolor='none')
    
    # xlim, ylim = np.percentile(tmp['ratio'], [1,99]), np.percentile(tmp['mCCCFrac'], [1,99]) 
    # ax.set_xlim([xlim[0]-0.001, xlim[1]+0.001])
    # ax.set_ylim([ylim[0]-0.001, ylim[1]+0.001])
    lim = [min([np.percentile(tmp['ratio'], 1), np.percentile(tmp['mCHFrac'], 1)]), 
           max([np.percentile(tmp['ratio'], 99), np.percentile(tmp['mCHFrac'], 99)])]
    diff = 0.05*(lim[1]-lim[0])
    ax.set_xlim([lim[0]-diff, lim[1]+diff])
    ax.set_ylim([lim[0]-diff, lim[1]+diff])
    ax.plot(lim, lim, c='k')
    ax.set_title(xx, fontsize=15)

for ax in axes[:,0]:
    ax.set_ylabel('Genome mCH/CH')

for ax in axes[-1]:
    ax.set_xlabel('Lambda mCH/CH')

plt.tight_layout()


In [ ]:
mch_df = meta.groupby('L1')[['mCHmC', 'mCHCov', 'mc', 'cov']].sum()
mch_df['LambdamCHFrac'] = mch_df['mc'] / mch_df['cov']
mch_df['BulkmCHFrac'] = mch_df['mCHmC'] / mch_df['mCHCov']
mch_df['AvemCHFrac'] = meta.groupby('L1')['mCHFrac'].mean()
mch_df.sort_values('AvemCHFrac') 

In [ ]:
from scipy.stats import binom
[binom.sf(m-1,n,p) for m,n,p in mch_df[['mCHmC', 'mCHCov', 'LambdamCHFrac']].values]

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(4,4), sharex=True, 
                         gridspec_kw={'height_ratios': [0.9,3]}, dpi=300)
fig.subplots_adjust(hspace=0.05)

# sns.scatterplot(mch_df, x='LambdamCHFrac', y='AvemCHFrac', hue='L1', palette='tab20')
leg = np.sort(mch_df.index)
colordict = {xx:yy for xx,yy in zip(leg, sns.color_palette('tab20', len(leg)))}
ylim = [[0.056, 0.061], [0.005, 0.021]]
d = .5  # proportion of vertical to horizontal extent of the slanted line
kwargs = dict(marker=[(-1, -d), (1, d)], markersize=12,
              linestyle="none", color='k', mec='k', mew=1, clip_on=False)

for k,ax in enumerate(axes):
    for xx,yy in zip(mch_df.index, mch_df[['LambdamCHFrac', 'AvemCHFrac']].values):
        ax.scatter(yy[0], yy[1], label=xx, c=colordict[xx], s=20, edgecolor='none')
        ax.set_ylim(ylim[k])
        if (yy[1]-yy[0]>0.003) & (yy[1]<ylim[k][1]) & (yy[1]>ylim[k][0]):
            ax.text(yy[0], yy[1], xx, ha='right', va='bottom')
    
# ax.scatter(mch_df['LambdamCHFrac'], mch_df['AvemCHFrac'], label=mch_df.index.astype(str), 
#            c=mch_df.index.astype(str).map(colordict), s=20, edgecolor='none')
# lim = [0.001, 0.065]
ax.set_xlim([0.005, 0.012])
ax.set_xticks(np.arange(0.006, 0.013, 0.002))
# ax.set_ylim(lim)
ax.plot(lim, lim, c='k')
ax.set_ylabel('Genome mCH/CH')
ax.set_xlabel('Lambda mCH/CH')
ax.spines.top.set_visible(False)
ax.xaxis.tick_bottom()
ax.plot([0, 1], [1, 1], transform=ax.transAxes, **kwargs)

ax = axes[0]
ax.set_yticks(np.arange(0.056, 0.061, 0.002))
ax.legend(loc='upper left', bbox_to_anchor=(1,1.05), ncol=2)
ax.spines.bottom.set_visible(False)
ax.xaxis.tick_top()
ax.tick_params(labeltop=False)  # don't put tick labels at the top
ax.plot([0, 1], [0, 0], transform=ax.transAxes, **kwargs)

# selc = (mch_df['AvemCHFrac'] - mch_df['LambdamCHFrac'] > 0.003) & (mch_df['AvemCHFrac'] < 0.025)
# print(selc.sum())
# for xx,coord in zip(mch_df.index[selc], mch_df[['LambdamCHFrac', 'AvemCHFrac']].values[selc]):
#     ax.text(coord[0], coord[1], xx, ha='right', va='bottom')
# plt.tight_layout()
plt.savefig('global_mCH_celltype.pdf', transparent=True)
